# Optimizing matrix multiplication with Numba — comparison & GPU techniques

This notebook continues directly from the previous session, where we sped up matrix multiplication step by step. The improvements here are about **measuring and comparing properly**, and about going deeper into **GPU optimization**:

- All version timings are recorded into a single **pandas table** so they are easy to compare side by side.
- We exclude **compilation time** from every measurement by doing a tiny **warm-up call** before timing the real run. We apply this uniformly to every JIT version so the comparison is fair. (We use warm-up rather than `cache=True` — which both `@njit` and `@cuda.jit` support — because warm-up cleanly isolates *compute* time and works identically for all versions.)
- We add two **optimized GPU versions** (explicit host/device copy, then shared-memory tiling), and then a section of **further techniques** with an honest verdict on which actually help this problem.

> Same workflow as before: for each version we go **Analyze → Design → Implement → Evaluate (runtime + correctness)**. Reminder from last time: on Colab, use the CUDA stack the platform already ships — don't pin old `numba`/`numba-cuda` versions, or you'll hit driver/toolkit mismatches.

---

## Environment check (Colab)

No installs needed — Colab already ships a recent, internally-consistent `numba` + `numba-cuda` (CUDA 12) stack. Just confirm the GPU is available. (If you ever break the environment, reset with **Runtime → Disconnect and delete runtime**, not a plain *Restart session*.)

In [ ]:
import numba
from numba import cuda
print('numba', numba.__version__)
print('CUDA available:', cuda.is_available())

### Generate the input data

We generate the two input matrices once and save them to disk, so every version reads the **same** input. `A` is `1000 x 1500`, `B` is `1500 x 2000`, giving `C = A @ B` of size `1000 x 2000`. Values are small integers, so the exact sums stay well within range (max magnitude `100*100*1500 = 15,000,000`).

In [ ]:
import numpy as np

np.random.seed(0)  # reproducible
A = np.random.randint(-100, 100, (1000, 1500))
B = np.random.randint(-100, 100, (1500, 2000))
np.savetxt('imat1.gz', A, fmt='%i')
np.savetxt('imat2.gz', B, fmt='%i')
print('A:', A.shape, '| B:', B.shape, '| C:', (A.shape[0], B.shape[1]))

In [ ]:
!ls -lh imat1.gz imat2.gz

### The comparison table

Every version appends its measured time to this pandas table, so at the end we can compare them all at a glance.

In [ ]:
import pandas as pd
all_results = pd.DataFrame(columns=['time (s)'], dtype=float)
all_results

---

## Sequential 1: NumPy's matrix-multiply operator

### Design
This version is one line, so we skip the design and go straight to implementation. NumPy's `@` calls an optimized, multi-threaded BLAS routine — our reference for both correctness and speed.

### Implementation

In [ ]:
%%writefile sequential1.py
import time
import numpy as np

A = np.loadtxt('imat1.gz', dtype=int)
B = np.loadtxt('imat2.gz', dtype=int)

start = time.perf_counter()
C = A @ B
end = time.perf_counter()
print(f'Processing time: {end - start} s')

np.savetxt('omat_s1.gz', C, fmt='%i')

### Evaluation

#### Runtime

In [ ]:
%%capture output
!python sequential1.py

In [ ]:
t = float(output.stdout.split()[-2])
all_results.loc['sequential 1: numpy @', 'time (s)'] = t
all_results

#### Correctness

This is essentially one line of trusted library code, so we treat its output as the **ground truth** that every later version is checked against.

In [ ]:
C_s1 = np.loadtxt('omat_s1.gz', dtype=int)

---

## Sequential 2: Numba `@njit` — compiled, sequential, on the CPU

### Design
*(File I/O is omitted from the design for clarity.)*

- **Input:** matrices `A`, `B`
- **Steps:** allocate the result `C`; call an `@njit` function that, for every row `r` and column `c` of `C`, sets `C[r, c]` to the dot product of row `r` of `A` with column `c` of `B`
- **Output:** `C`

We use `@njit` (`= @jit(nopython=True)`; since Numba 0.59 plain `@jit` defaults to the same). Note the **warm-up call** on a tiny dummy array: it forces compilation *before* the timer starts, so we measure pure execution.

### Implementation

In [ ]:
%%writefile sequential2.py
import time
import numpy as np
from numba import njit

@njit
def mul_mat(A, B, C):
    for r in range(C.shape[0]):
        for c in range(C.shape[1]):
            temp = 0
            for i in range(A.shape[1]):
                temp += A[r, i] * B[i, c]
            C[r, c] = temp

# Warm-up: compile now, on a tiny int array, so timing below excludes compilation
dummy = np.empty((1, 1), dtype=int)
mul_mat(dummy, dummy, dummy)

A = np.loadtxt('imat1.gz', dtype=int)
B = np.loadtxt('imat2.gz', dtype=int)

start = time.perf_counter()
C = np.empty((A.shape[0], B.shape[1]), dtype=int)
mul_mat(A, B, C)
end = time.perf_counter()
print(f'Processing time: {end - start} s')

np.savetxt('omat_s2.gz', C, fmt='%i')

> **Why warm-up matters:** the *first* call to a Numba function includes compilation. Timing it would "benchmark the compiler." By calling once on dummy data first, the timed call runs already-compiled machine code. We do this in every version so all timings measure the same thing: compute only.

### Evaluation

#### Runtime

In [ ]:
%%capture output
!python sequential2.py

In [ ]:
t = float(output.stdout.split()[-2])
all_results.loc['sequential 2: @njit (CPU)', 'time (s)'] = t
all_results

#### Correctness

In [ ]:
C_s2 = np.loadtxt('omat_s2.gz', dtype=int)
print('mean abs diff vs baseline:', np.mean(np.abs(C_s2 - C_s1)))

---

## Parallel 1: Numba `@njit(parallel=True)` — parallel on the CPU

Our project focuses on the GPU; this version is here mainly to show Numba can *also* parallelize across CPU cores with almost no extra code.

### Analysis
This program has a single processing step (the multiply), so that is what we accelerate. It parallelizes well because **every output element is computed independently**.

### Design

In [ ]:
!lscpu | grep -E 'Model name|^CPU\(s\)|Thread|Core'

Each `C[r, c]` is independent, so the work can be split across threads. With Numba you only mark which loops are independent (using `prange`); Numba distributes those iterations across threads automatically.

### Implementation

In [ ]:
%%writefile parallel1.py
import time
import numpy as np
from numba import njit, prange
from numba import config
config.THREADING_LAYER = 'omp'

@njit(parallel=True)
def mul_mat(A, B, C):
    for r in prange(C.shape[0]):   # prange: iterations are independent -> run in parallel
        for c in range(C.shape[1]):
            temp = 0
            for i in range(A.shape[1]):
                temp += A[r, i] * B[i, c]
            C[r, c] = temp

dummy = np.empty((1, 1), dtype=int)  # warm-up
mul_mat(dummy, dummy, dummy)

A = np.loadtxt('imat1.gz', dtype=int)
B = np.loadtxt('imat2.gz', dtype=int)

start = time.perf_counter()
C = np.empty((A.shape[0], B.shape[1]), dtype=int)
mul_mat(A, B, C)
end = time.perf_counter()
print(f'Processing time: {end - start} s')

np.savetxt('omat_p1.gz', C, fmt='%i')

> **Tip:** parallelize the **outer** loop (`prange` on `r`). Each thread then owns whole rows of `C`, keeping the work per thread large. The speedup is bounded by core/thread count and memory bandwidth (Amdahl's law), so don't expect it to scale to the full thread count.

### Evaluation

#### Runtime

In [ ]:
%%capture output
!python parallel1.py

In [ ]:
t = float(output.stdout.split()[-2])
all_results.loc['parallel 1: @njit(parallel=True) (CPU)', 'time (s)'] = t
all_results

#### Correctness

In [ ]:
C_p1 = np.loadtxt('omat_p1.gz', dtype=int)
print('mean abs diff vs baseline:', np.mean(np.abs(C_p1 - C_s1)))

---

## Parallel 2: Numba `@cuda.jit` — parallel on the GPU

This is the kind of parallelism the project focuses on.

### Analysis
Same single step, same independence between output elements — but the GPU has a very different execution model.

### Design: the thread / block / grid model
We launch **thousands of threads**, each running the same *kernel*, with **one thread per output element** `C[r, c]`. Threads are grouped into a 2D **block**, and blocks into a 2D **grid**, mapped onto the matrix:

```text
          GRID of blocks                  one thread -> one C[r, c]
  +---------+---------+---------+
  | block   | block   | block   |    block = 32 x 32 threads
  | 32x32   | 32x32   |   ...   |    grid.x  -> columns (c)
  +---------+---------+---------+    grid.y  -> rows    (r)
  | block   | block   | block   |
  +---------+---------+---------+
```

A thread finds its `(r, c)` from its grid position. `cuda.grid(2)` returns `(x, y)`; we map **x → column** and **y → row**, so we write `c, r = cuda.grid(2)`. Because the grid is rounded up, more threads are launched than elements, so every kernel must **check its bounds**.

[Illustration of the grid/block layout](https://docs.google.com/spreadsheets/d/13upl8S0wBESvKUIZVO98-f5JD6rA4Mfyvepw_bDAHoc/edit?usp=sharing)

> The choice **x → column** is not arbitrary: it is what makes global-memory access *coalesced* (consecutive threads touch consecutive addresses). We measure exactly why this matters in the "further techniques" section.

### Implementation

In [ ]:
%%writefile parallel2.py
import time
import math
import numpy as np
from numba import cuda

@cuda.jit  # the code ONE GPU thread runs (add cache=True to enable on-disk caching)
def mul_mat_kernel(A, B, C):
    c, r = cuda.grid(2)  # (x, y) -> (column, row)
    if r < C.shape[0] and c < C.shape[1]:
        temp = 0
        for i in range(A.shape[1]):
            temp += A[r, i] * B[i, c]
        C[r, c] = temp

dummy = np.empty((1, 1), dtype=int)  # warm-up (compile the kernel)
mul_mat_kernel[1, 1](dummy, dummy, dummy)

A = np.loadtxt('imat1.gz', dtype=int)
B = np.loadtxt('imat2.gz', dtype=int)

start = time.perf_counter()
C = np.empty((A.shape[0], B.shape[1]), dtype=int)
block_size = (32, 32)  # 1024 threads/block
grid_size = (math.ceil(C.shape[1] / block_size[0]),   # x -> columns
             math.ceil(C.shape[0] / block_size[1]))   # y -> rows
mul_mat_kernel[grid_size, block_size](A, B, C)
cuda.synchronize()
end = time.perf_counter()
print(f'Processing time: {end - start} s')

np.savetxt('omat_p2.gz', C, fmt='%i')

> **Implicit data transfer:** we pass plain NumPy (host) arrays straight to the kernel, so Numba automatically copies `A`, `B` to the GPU and copies `C` back. Convenient, but that transfer is hidden inside the timing and happens on *every* call. The next version makes it explicit. (`cuda.synchronize()` is needed because kernel launches are asynchronous — without it we'd stop the timer before the GPU finished.)

### Evaluation

#### Runtime

In [ ]:
%%capture output
!python parallel2.py

In [ ]:
t = float(output.stdout.split()[-2])
all_results.loc['parallel 2: @cuda.jit (implicit copy)', 'time (s)'] = t
all_results

#### Correctness

In [ ]:
C_p2 = np.loadtxt('omat_p2.gz', dtype=int)
print('mean abs diff vs baseline:', np.mean(np.abs(C_p2 - C_s1)))

---

## Parallel optimized 1: explicit host/device memory management

### Analysis: where does the time go?

To optimize, first measure. There are three ways to profile GPU code on Colab, from simplest to most powerful:

1. **CUDA events** — the most portable method. No external tool, works everywhere. We time the three phases (H2D copy, kernel, D2H copy) directly in Python.
2. **`nvprof`** — the classic NVIDIA profiler. Deprecated, but still works on Colab when you get a **Tesla T4** (compute capability 7.5). Fails on newer GPUs like A100 or L4 (CC ≥ 8.0). Produces the clearest per-operation trace.
3. **Nsight Systems (`nsys`)** — the modern replacement. Works on **all** GPUs. Needs a one-time install on Colab.

We show all three below so you can use whichever fits your GPU.

#### Method 1: CUDA events (always works)

In [ ]:
%%writefile profile_phases.py
import math
import numpy as np
from numba import cuda

@cuda.jit
def mul_mat_kernel(A, B, C):
    c, r = cuda.grid(2)
    if r < C.shape[0] and c < C.shape[1]:
        temp = 0
        for i in range(A.shape[1]):
            temp += A[r, i] * B[i, c]
        C[r, c] = temp

dummy = np.empty((1, 1), dtype=int)
mul_mat_kernel[1, 1](dummy, dummy, dummy)  # warm-up

A = np.loadtxt('imat1.gz', dtype=int)
B = np.loadtxt('imat2.gz', dtype=int)
block_size = (32, 32)
grid_size = (math.ceil(B.shape[1] / block_size[0]),
             math.ceil(A.shape[0] / block_size[1]))

e0, e1, e2, e3 = cuda.event(), cuda.event(), cuda.event(), cuda.event()

e0.record()
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array((A.shape[0], B.shape[1]), dtype=int)
e1.record()
mul_mat_kernel[grid_size, block_size](d_A, d_B, d_C)
e2.record()
C = d_C.copy_to_host()
e3.record()
e3.synchronize()

print(f'H2D copy : {cuda.event_elapsed_time(e0, e1):8.3f} ms')
print(f'Kernel   : {cuda.event_elapsed_time(e1, e2):8.3f} ms')
print(f'D2H copy : {cuda.event_elapsed_time(e2, e3):8.3f} ms')

In [ ]:
!python profile_phases.py

#### Method 2: `nvprof` (works on T4 / CC ≤ 7.5 only)

`nvprof --print-gpu-trace` lists every GPU operation chronologically — each memory copy and each kernel launch as a separate row with size, duration, and direction. This is the clearest view for understanding implicit copy behavior. If Colab assigned you a newer GPU (A100, L4), this cell will fail — skip it and use `nsys` below.

In [ ]:
!nvprof --print-gpu-trace python parallel2.py

#### Method 3: Nsight Systems — `nsys` (modern, works on all GPUs)

Nsight Systems is the modern replacement for `nvprof`. It needs a one-time install on Colab (doesn't touch your CUDA toolkit or driver). The `--report cuda_gpu_trace` flag gives the same per-operation chronological trace as `nvprof --print-gpu-trace`.

In [ ]:
!apt-get update -qq
!apt-get install -y cuda-nsight-systems-13-3 -qq
!nsys --version

In [ ]:
!nsys profile --trace=cuda --cuda-memory-usage=true --force-overwrite=true \
      -o p2_report python parallel2.py

In [ ]:
# Per-operation trace (equivalent to nvprof --print-gpu-trace)
!nsys stats --report cuda_gpu_trace p2_report.nsys-rep

#### How to read the profiler output

Whether you used `nvprof` or `nsys`, the trace shows the same two groups of operations: the **warm-up call** (tiny 1×1 dummy) and the **real call** (full-size matrices). Focus on the real call. You should see something like this sequence:

```
HtoD  11.4 MB   ~2.8 ms   ← A copied to device
HtoD  22.9 MB   ~5.0 ms   ← B copied to device
HtoD  15.3 MB   ~3.3 ms   ← C copied to device — UNNECESSARY (C is output-only)
Kernel           ~53  ms   ← mul_mat_kernel — the actual compute
DtoH  11.4 MB   ~2.4 ms   ← A copied back — UNNECESSARY (A is input-only)
DtoH  22.9 MB   ~5.0 ms   ← B copied back — UNNECESSARY (B is input-only)
DtoH  15.3 MB  ~13.4 ms   ← C copied back — the only copy we need
```

The three matrices total ~49.6 MB (A: 11.4 + B: 22.9 + C: 15.3). Numba's implicit copy transfers **all three both ways** — 104 MB of transfers for 49.6 MB of actual data need. Only A and B need to go *to* the device, and only C needs to come *back* — that's 49.6 MB instead of 104 MB.

Other useful columns in the trace:
- **Regs** — registers used per thread (42 in our kernel; more registers = less threads per SM = lower occupancy)
- **SSMem / DSMem** — static/dynamic shared memory per block (0 B here; changes in the tiled version)
- **Grid Size / Block Size** — confirms our `(63, 32, 1)` grid and `(32, 32, 1)` block layout
- **Throughput** — effective bandwidth of each copy (compare against the GPU's theoretical PCIe bandwidth)

> **Tip:** to view the `.nsys-rep` file visually, download it from Colab and open it in the **Nsight Systems GUI** on your computer (free from https://developer.nvidia.com/nsight-systems/get-started — no GPU needed on the viewing machine). The timeline shows H2D copies (green), kernel execution (blue), and D2H copies (red).

> **Why not Nsight Compute (`ncu`)?** That's the tool that would *quantify* kernel-level metrics (coalescing efficiency, bank conflicts, occupancy, roofline). But it reads GPU hardware performance counters, which have been permission-gated since a 2018 driver security change. On Colab `ncu` fails with `ERR_NVGPUCTRPERM` — use it on hardware you control.

Two things this reveals (and motivates the optimizations):

1. In `parallel2` the implicit transfer copies all of `A`, `B`, `C` to the device and `C` back **every call**. In a real program you transfer once and reuse the device arrays, so that cost should be separated out and, where possible, avoided.
2. The **kernel** dominates — so after fixing the obvious transfer waste, real optimization effort belongs in the kernel (that's the shared-memory version next).

In this version we manage the device memory ourselves: copy inputs to the device *before* the timed region, allocate the output on the device, run, then copy the result back. The timer now reflects compute (plus the copies we explicitly chose to include), not Numba's hidden round-trips.

### Design
- Copy `A`, `B` from host to `d_A`, `d_B` on the device
- Allocate `d_C` on the device
- Launch the kernel on `d_A`, `d_B`, `d_C`
- Copy `d_C` back to host `C`

### Implementation

In [ ]:
%%writefile parallel_optimized1.py
import time
import math
import numpy as np
from numba import cuda

@cuda.jit
def mul_mat_kernel(A, B, C):
    c, r = cuda.grid(2)
    if r < C.shape[0] and c < C.shape[1]:
        temp = 0
        for i in range(A.shape[1]):
            temp += A[r, i] * B[i, c]
        C[r, c] = temp

dummy = np.empty((1, 1), dtype=int)  # warm-up
mul_mat_kernel[1, 1](dummy, dummy, dummy)

A = np.loadtxt('imat1.gz', dtype=int)
B = np.loadtxt('imat2.gz', dtype=int)

start = time.perf_counter()
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array((A.shape[0], B.shape[1]), dtype=int)
block_size = (32, 32)
grid_size = (math.ceil(d_C.shape[1] / block_size[0]),
             math.ceil(d_C.shape[0] / block_size[1]))
mul_mat_kernel[grid_size, block_size](d_A, d_B, d_C)
C = d_C.copy_to_host()  # copy_to_host synchronizes
end = time.perf_counter()
print(f'Processing time: {end - start} s')

np.savetxt('omat_po1.gz', C, fmt='%i')

### Evaluation

#### Runtime

In [ ]:
%%capture output
!python parallel_optimized1.py

In [ ]:
t = float(output.stdout.split()[-2])
all_results.loc['parallel opt 1: @cuda.jit + explicit copy', 'time (s)'] = t
all_results

#### Correctness

In [ ]:
C_po1 = np.loadtxt('omat_po1.gz', dtype=int)
print('mean abs diff vs baseline:', np.mean(np.abs(C_po1 - C_s1)))

---

## Parallel optimized 2: shared-memory tiling

### Analysis
We now optimize the kernel itself, since it dominates the time. In `parallel2`, `A` and `B` live in **global memory (GMEM)** — large but slow, physically in the device's DRAM far from the cores. Worse, each element of `A`/`B` is re-read from GMEM by *many* threads in a block.

We can cut the number of slow GMEM reads using **shared memory (SMEM)** — a small, fast, per-block memory close to the cores:

- Each block first cooperatively loads the slice of `A`/`B` it needs from GMEM into SMEM (each element read once), then all threads in the block read from fast SMEM.
- That slice is usually too big for SMEM, so we process it in **tiles**: load a 32×32 tile of `A` and of `B` into SMEM, accumulate partial results, then load the next tile (overwriting the previous), and so on. This is the classic *tiled matrix multiply*.

(If you haven't taken a parallel-programming course, the GPU memory hierarchy — registers, shared memory, global memory — is the key background here.)

### Design
- Allocate `C`; launch a tiled `@cuda.jit` kernel that, tile by tile, loads `A`/`B` into SMEM, synchronizes, accumulates, synchronizes, and repeats.
- [Illustration of the tiling and thread cooperation](https://docs.google.com/spreadsheets/d/1Hl6eeCJehwH9iJbF21M4F7go_jFOiL-v5jcO-_6qKA0/edit?usp=sharing)

### Implementation
For simplicity we fix the block size at 32×32 and declare the SMEM tiles statically at that size.

In [ ]:
%%writefile parallel_optimized2.py
import time
import math
import numpy as np
from numba import cuda

TILE = 32

@cuda.jit
def mul_mat_kernel(A, B, C):
    # One s_A and one s_B tile per block, in shared memory
    s_A = cuda.shared.array((TILE, TILE), dtype=A.dtype)
    s_B = cuda.shared.array((TILE, TILE), dtype=A.dtype)

    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    c, r = cuda.grid(2)

    temp = 0
    for phase in range(math.ceil(A.shape[1] / TILE)):
        # Cooperatively load one tile of A and one tile of B into SMEM
        if r < A.shape[0] and phase * TILE + tx < A.shape[1]:
            s_A[ty, tx] = A[r, phase * TILE + tx]
        else:
            s_A[ty, tx] = 0
        if phase * TILE + ty < B.shape[0] and c < B.shape[1]:
            s_B[ty, tx] = B[phase * TILE + ty, c]
        else:
            s_B[ty, tx] = 0

        cuda.syncthreads()  # all threads must finish loading before we read the tile

        for i in range(TILE):
            temp += s_A[ty, i] * s_B[i, tx]

        cuda.syncthreads()  # all threads must finish using the tile before the next overwrites it

    if r < C.shape[0] and c < C.shape[1]:
        C[r, c] = temp

dummy = np.empty((1, 1), dtype=int)  # warm-up
mul_mat_kernel[(1, 1), (TILE, TILE)](dummy, dummy, dummy)

A = np.loadtxt('imat1.gz', dtype=int)
B = np.loadtxt('imat2.gz', dtype=int)

start = time.perf_counter()
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array((A.shape[0], B.shape[1]), dtype=int)
block_size = (TILE, TILE)
grid_size = (math.ceil(d_C.shape[1] / block_size[0]),
             math.ceil(d_C.shape[0] / block_size[1]))
mul_mat_kernel[grid_size, block_size](d_A, d_B, d_C)
C = d_C.copy_to_host()
end = time.perf_counter()
print(f'Processing time: {end - start} s')

np.savetxt('omat_po2.gz', C, fmt='%i')

> **Why the two `syncthreads()`:** the first makes sure the whole tile is loaded into SMEM before any thread reads it; the second makes sure every thread has finished using the tile before the next phase overwrites it. Removing either causes a race and wrong results.

### Evaluation

#### Runtime

In [ ]:
%%capture output
!python parallel_optimized2.py

In [ ]:
t = float(output.stdout.split()[-2])
all_results.loc['parallel opt 2: + shared-memory tiling', 'time (s)'] = t
all_results

#### Correctness

In [ ]:
C_po2 = np.loadtxt('omat_po2.gz', dtype=int)
print('mean abs diff vs baseline:', np.mean(np.abs(C_po2 - C_s1)))

---

# Going further: more GPU optimization techniques

Below is each technique with an **honest verdict** for *this* matmul, the concept, and code where it pays off. Not every well-known CUDA trick helps every problem — knowing which to reach for is half the skill.

## A. Coalesced global-memory access  —  verdict: ESSENTIAL (already applied)

When the 32 threads of a *warp* read global memory, the hardware is fastest if they touch **consecutive addresses** — it then services them in one transaction ("coalesced"). If they touch scattered addresses, it issues many transactions (slow).

In `parallel2`, threads in a warp have consecutive `threadIdx.x`, and we mapped **x → column (`c`)**. So:
- `B[i, c]` for consecutive `c` → consecutive memory (row-major) → **coalesced** ✓
- `C[r, c]` for consecutive `c` → **coalesced** ✓
- `A[r, i]` is the same address for all threads in the warp → **broadcast** (fine) ✓

To prove it matters, here is the *same* kernel with the mapping flipped to **x → row** — still correct, but now consecutive threads write `C[r, c]` and read `A[r, i]` with a large stride, so the accesses are **not coalesced**. Watch it run slower:

In [ ]:
%%writefile parallel_noncoalesced.py
import time
import math
import numpy as np
from numba import cuda

@cuda.jit
def mul_mat_kernel(A, B, C):
    r, c = cuda.grid(2)  # x -> ROW (the slow choice): de-coalesces A reads and C writes
    if r < C.shape[0] and c < C.shape[1]:
        temp = 0
        for i in range(A.shape[1]):
            temp += A[r, i] * B[i, c]
        C[r, c] = temp

dummy = np.empty((1, 1), dtype=int)
mul_mat_kernel[1, 1](dummy, dummy, dummy)  # warm-up

A = np.loadtxt('imat1.gz', dtype=int)
B = np.loadtxt('imat2.gz', dtype=int)

start = time.perf_counter()
d_A = cuda.to_device(A); d_B = cuda.to_device(B)
d_C = cuda.device_array((A.shape[0], B.shape[1]), dtype=int)
block_size = (32, 32)
grid_size = (math.ceil(d_C.shape[0] / block_size[0]),   # x dim now spans ROWS
             math.ceil(d_C.shape[1] / block_size[1]))   # y dim spans COLUMNS
mul_mat_kernel[grid_size, block_size](d_A, d_B, d_C)
C = d_C.copy_to_host()
end = time.perf_counter()
print(f'Processing time: {end - start} s')
np.savetxt('omat_nc.gz', C, fmt='%i')

In [ ]:
%%capture output
!python parallel_noncoalesced.py

In [ ]:
t = float(output.stdout.split()[-2])
all_results.loc['ANTI-PATTERN: non-coalesced access', 'time (s)'] = t
C_nc = np.loadtxt('omat_nc.gz', dtype=int)
print('correct?', np.mean(np.abs(C_nc - C_s1)) == 0)
all_results

Same result, noticeably slower. The lesson: on the GPU, *how* you map threads to data can matter as much as the algorithm. Our `c, r = cuda.grid(2)` choice was a coalescing optimization all along.


## B. Use 32-bit types when you can (int64 → int32)  —  verdict: EASY WIN

GPUs have far more 32-bit arithmetic throughput than 64-bit. Our values fit easily in `int32` (max magnitude 15,000,000 ≪ 2,147,483,647), so switching `int64 → int32` halves memory traffic and uses the faster 32-bit units — for free. Here is the tiled kernel in `int32`:

In [ ]:
%%writefile parallel_optimized3_int32.py
import time
import math
import numpy as np
from numba import cuda

TILE = 32

@cuda.jit
def mul_mat_kernel(A, B, C):
    s_A = cuda.shared.array((TILE, TILE), dtype=A.dtype)
    s_B = cuda.shared.array((TILE, TILE), dtype=A.dtype)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    c, r = cuda.grid(2)
    temp = 0
    for phase in range(math.ceil(A.shape[1] / TILE)):
        if r < A.shape[0] and phase * TILE + tx < A.shape[1]:
            s_A[ty, tx] = A[r, phase * TILE + tx]
        else:
            s_A[ty, tx] = 0
        if phase * TILE + ty < B.shape[0] and c < B.shape[1]:
            s_B[ty, tx] = B[phase * TILE + ty, c]
        else:
            s_B[ty, tx] = 0
        cuda.syncthreads()
        for i in range(TILE):
            temp += s_A[ty, i] * s_B[i, tx]
        cuda.syncthreads()
    if r < C.shape[0] and c < C.shape[1]:
        C[r, c] = temp

dummy = np.empty((1, 1), dtype=np.int32)  # warm-up (note: int32)
mul_mat_kernel[(1, 1), (TILE, TILE)](dummy, dummy, dummy)

A = np.loadtxt('imat1.gz', dtype=np.int32)  # load as int32
B = np.loadtxt('imat2.gz', dtype=np.int32)

start = time.perf_counter()
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array((A.shape[0], B.shape[1]), dtype=np.int32)
block_size = (TILE, TILE)
grid_size = (math.ceil(d_C.shape[1] / block_size[0]),
             math.ceil(d_C.shape[0] / block_size[1]))
mul_mat_kernel[grid_size, block_size](d_A, d_B, d_C)
C = d_C.copy_to_host()
end = time.perf_counter()
print(f'Processing time: {end - start} s')
np.savetxt('omat_po3.gz', C, fmt='%i')

In [ ]:
%%capture output
!python parallel_optimized3_int32.py

In [ ]:
t = float(output.stdout.split()[-2])
all_results.loc['parallel opt 3: tiling + int32', 'time (s)'] = t
C_po3 = np.loadtxt('omat_po3.gz', dtype=int)
print('correct?', np.mean(np.abs(C_po3 - C_s1)) == 0)
all_results

> Same idea applies to `float32` vs `float64` in floating-point code, where you can also add `fastmath=True` to let the compiler use faster, slightly-less-precise math. Always confirm the reduced precision is acceptable.


## C. Shared-memory bank conflicts  —  verdict: matters for SMEM kernels (minor for our matmul)

Shared memory is split into 32 **banks**. A warp runs at full speed when its 32 threads hit 32 *different* banks (or all read the same address = broadcast). When several threads hit *different* addresses in the *same* bank, the accesses **serialize** — a "bank conflict." A classic trigger is a **column-wise** access with stride 32, because element `k` of every row lands in the same bank.

In our tiled matmul inner loop:
- `s_B[i, tx]` — threads vary `tx` over `0..31` → consecutive elements of a row → 32 different banks → **conflict-free** ✓
- `s_A[ty, i]` — for fixed `i`, a warp shares `ty` → same address → **broadcast** ✓

So our matmul is already mostly conflict-free. We'll still apply the standard fix — **pad the tile by one column** so the column stride becomes 33 and no longer aligns to a bank — to show *how* it's written, then demonstrate it on a kernel that genuinely suffers (a transpose), where the speedup is real.

### C.1 Padding applied to our tiled matmul (concrete, in-context)

Identical to `parallel optimized 2`, except the shared tiles are declared `(32, 33)` instead of `(32, 32)`. The extra column is just padding; indexing is unchanged. Expect roughly the *same* time here — the point is the technique, not a speedup for this already-conflict-free pattern.

In [ ]:
%%writefile parallel_optimized2_padded.py
import time
import math
import numpy as np
from numba import cuda

TILE = 32
PAD = 33  # = TILE + 1; must be a plain literal for cuda.shared.array

@cuda.jit
def mul_mat_kernel(A, B, C):
    # Padded shared tiles: (32, 33). The last column is unused padding.
    s_A = cuda.shared.array((TILE, PAD), dtype=A.dtype)
    s_B = cuda.shared.array((TILE, PAD), dtype=A.dtype)

    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    c, r = cuda.grid(2)

    temp = 0
    for phase in range(math.ceil(A.shape[1] / TILE)):
        if r < A.shape[0] and phase * TILE + tx < A.shape[1]:
            s_A[ty, tx] = A[r, phase * TILE + tx]
        else:
            s_A[ty, tx] = 0
        if phase * TILE + ty < B.shape[0] and c < B.shape[1]:
            s_B[ty, tx] = B[phase * TILE + ty, c]
        else:
            s_B[ty, tx] = 0

        cuda.syncthreads()

        for i in range(TILE):
            temp += s_A[ty, i] * s_B[i, tx]

        cuda.syncthreads()

    if r < C.shape[0] and c < C.shape[1]:
        C[r, c] = temp

dummy = np.empty((1, 1), dtype=int)  # warm-up
mul_mat_kernel[(1, 1), (TILE, TILE)](dummy, dummy, dummy)

A = np.loadtxt('imat1.gz', dtype=int)
B = np.loadtxt('imat2.gz', dtype=int)

start = time.perf_counter()
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array((A.shape[0], B.shape[1]), dtype=int)
block_size = (TILE, TILE)
grid_size = (math.ceil(d_C.shape[1] / block_size[0]),
             math.ceil(d_C.shape[0] / block_size[1]))
mul_mat_kernel[grid_size, block_size](d_A, d_B, d_C)
C = d_C.copy_to_host()
end = time.perf_counter()
print(f'Processing time: {end - start} s')
np.savetxt('omat_po2pad.gz', C, fmt='%i')

In [ ]:
%%capture output
!python parallel_optimized2_padded.py

In [ ]:
t = float(output.stdout.split()[-2])
all_results.loc['parallel opt 2b: tiling + padded SMEM', 'time (s)'] = t
C_po2pad = np.loadtxt('omat_po2pad.gz', dtype=int)
print('correct?', np.mean(np.abs(C_po2pad - C_s1)) == 0)
all_results

### C.2 Seeing a bank conflict for real: shared-memory transpose

Matrix **transpose** is the textbook case. A tiled transpose loads a block into shared memory row-wise (no conflict) but writes it out **column-wise** (`tile[tx, ty]`), which is a 32-way bank conflict. Padding the tile to `(32, 33)` removes it. The script below runs **both** kernels on the same data and prints both times, so you can see the difference directly.

In [ ]:
%%writefile transpose_bankconflict.py
import numpy as np
import math
from numba import cuda

TILE = 32
PAD = 33  # = TILE + 1; must be a plain literal for cuda.shared.array

@cuda.jit
def transpose_conflict(inp, out):
    tile = cuda.shared.array((TILE, TILE), dtype=inp.dtype)   # NOT padded
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    x = cuda.blockIdx.x * TILE + tx
    y = cuda.blockIdx.y * TILE + ty
    if x < inp.shape[1] and y < inp.shape[0]:
        tile[ty, tx] = inp[y, x]          # row-wise write: no conflict
    cuda.syncthreads()
    x = cuda.blockIdx.y * TILE + tx        # transposed block coordinates
    y = cuda.blockIdx.x * TILE + ty
    if x < out.shape[1] and y < out.shape[0]:
        out[y, x] = tile[tx, ty]          # column-wise read: 32-way BANK CONFLICT

@cuda.jit
def transpose_padded(inp, out):
    tile = cuda.shared.array((TILE, PAD), dtype=inp.dtype)  # padded: +1 column
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    x = cuda.blockIdx.x * TILE + tx
    y = cuda.blockIdx.y * TILE + ty
    if x < inp.shape[1] and y < inp.shape[0]:
        tile[ty, tx] = inp[y, x]
    cuda.syncthreads()
    x = cuda.blockIdx.y * TILE + tx
    y = cuda.blockIdx.x * TILE + ty
    if x < out.shape[1] and y < out.shape[0]:
        out[y, x] = tile[tx, ty]          # same logic, but padding removes the conflict

def time_kernel(kernel, d_in, d_out, grid, block, repeat=20):
    kernel[grid, block](d_in, d_out)      # warm-up / compile
    cuda.synchronize()
    e0, e1 = cuda.event(), cuda.event()
    e0.record()
    for _ in range(repeat):
        kernel[grid, block](d_in, d_out)
    e1.record(); e1.synchronize()
    return cuda.event_elapsed_time(e0, e1) / repeat  # ms per run

N = 4096                                   # big square matrix so the effect is visible
inp = np.random.randint(-100, 100, (N, N)).astype(np.int32)
d_in = cuda.to_device(inp)
d_out = cuda.device_array((N, N), dtype=np.int32)
block = (TILE, TILE)
grid = (math.ceil(N / TILE), math.ceil(N / TILE))

t_conflict = time_kernel(transpose_conflict, d_in, d_out, grid, block)
out_conflict = d_out.copy_to_host()

t_padded = time_kernel(transpose_padded, d_in, d_out, grid, block)
out_padded = d_out.copy_to_host()

ok = np.array_equal(out_conflict, inp.T) and np.array_equal(out_padded, inp.T)
print(f'conflicted (no padding): {t_conflict:.4f} ms')
print(f'padded   (32x33 tile)  : {t_padded:.4f} ms')
print(f'speedup from padding   : {t_conflict / t_padded:.2f}x')
print(f'both correct           : {ok}')

In [ ]:
!python transpose_bankconflict.py

You should see the padded transpose run meaningfully faster than the conflicted one (often ~1.3–2x, hardware-dependent), with both producing the correct transpose. That gap *is* the bank-conflict penalty. Back in our matmul, the access pattern never created this conflict in the first place — which is why the padded matmul above was only about as fast as the unpadded one. The rule: reach for tile padding the moment a profiler reports shared-memory bank conflicts, typically in transpose- or column-heavy kernels.

## D. Thread coarsening (more work per thread)  —  verdict: standard next step

Right now each thread computes exactly one `C[r, c]`. **Thread coarsening** has each thread compute *several* output elements, so values it already loaded into registers are reused across those outputs. This raises *arithmetic intensity* (math per byte loaded) and is how high-performance matmul kernels close the gap toward cuBLAS.

### The idea

With `COARSEN = 4`, each thread computes **4 columns** of `C` instead of 1. The columns are spaced `blockDim.x` apart so that consecutive threads still touch consecutive memory — keeping the access **coalesced**.

```text
Thread tx=0 computes:  C[r, base],  C[r, base+32],  C[r, base+64],  C[r, base+96]
Thread tx=1 computes:  C[r, base+1], C[r, base+33], C[r, base+65], C[r, base+97]
  ...                                                   (still coalesced within each group)
```

The key benefit: in the inner loop, `A[r, i]` is loaded from global memory **once** and reused across all `COARSEN` output columns. Without coarsening, that same value would be loaded separately by `COARSEN` different threads — wasting bandwidth.

### Lineage

This version is built on top of **parallel opt 1** (the naive kernel with explicit device memory, no shared-memory tiling). It reads `A` and `B` directly from global memory — the only change is the coarsening loop. The fair comparison is therefore against opt 1, not against the tiled versions.

### Implementation

In [ ]:
%%writefile parallel_coarsened.py
import time
import math
import numpy as np
from numba import cuda, int64

COARSEN = 4

@cuda.jit
def mul_mat_coarsened(A, B, C):
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    r = cuda.blockIdx.y * cuda.blockDim.y + ty
    # Each thread handles COARSEN columns, spaced blockDim.x apart (stays coalesced)
    base_c = cuda.blockIdx.x * cuda.blockDim.x * COARSEN + tx

    # One accumulator per output element, stored in registers (local array)
    acc = cuda.local.array(4, dtype=int64)  # literal 4 must match COARSEN
    for k in range(COARSEN):
        acc[k] = 0

    if r < C.shape[0]:
        for i in range(A.shape[1]):
            a_val = A[r, i]                        # loaded ONCE from global memory
            for k in range(COARSEN):
                c = base_c + k * cuda.blockDim.x
                if c < C.shape[1]:
                    acc[k] += a_val * B[i, c]      # reused COARSEN times

        for k in range(COARSEN):
            c = base_c + k * cuda.blockDim.x
            if c < C.shape[1]:
                C[r, c] = acc[k]

# Warm-up
dummy = np.empty((1, 1), dtype=int)
mul_mat_coarsened[(1, 1), (1, 1)](dummy, dummy, dummy)

A = np.loadtxt('imat1.gz', dtype=int)
B = np.loadtxt('imat2.gz', dtype=int)

start = time.perf_counter()
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array((A.shape[0], B.shape[1]), dtype=int)
block_size = (32, 32)
# Grid x is divided by COARSEN — each thread covers COARSEN columns
grid_size = (math.ceil(d_C.shape[1] / (block_size[0] * COARSEN)),
             math.ceil(d_C.shape[0] / block_size[1]))
mul_mat_coarsened[grid_size, block_size](d_A, d_B, d_C)
C = d_C.copy_to_host()
end = time.perf_counter()
print(f'Processing time: {end - start} s')

np.savetxt('omat_coarse.gz', C, fmt='%i')

> **Why `cuda.local.array(4, dtype=int64)`?** The accumulators live in per-thread **registers** (fast, private). The size must be a plain literal (`4`, not `COARSEN`) — same gotcha as `cuda.shared.array`. We import `int64` from `numba` because `cuda.local.array` needs a Numba type, not a NumPy dtype.

### Evaluation

#### Runtime

In [ ]:
%%capture output
!python parallel_coarsened.py

In [ ]:
t = float(output.stdout.split()[-2])
all_results.loc['parallel coarsened: opt1 + 4 cols/thread', 'time (s)'] = t
all_results

#### Correctness

In [ ]:
C_coarse = np.loadtxt('omat_coarse.gz', dtype=int)
print('mean abs diff vs baseline:', np.mean(np.abs(C_coarse - C_s1)))

### What coarsening changes under the hood

| | Without coarsening | With `COARSEN = 4` |
|---|---|---|
| Threads per output element | 1 | 1 |
| Output elements per thread | 1 | 4 |
| Grid x dimension | `ceil(2000 / 32) = 63` | `ceil(2000 / 128) = 16` |
| `A[r, i]` global loads per thread | 1 per iteration | 1 per iteration (reused 4×) |
| Total `A[r, i]` loads across grid | 1 per (thread, i) | **4× fewer** |

The grid shrinks (fewer blocks launched), but each thread does more useful work per global-memory load.

### Honest comparison: coarsening vs tiling

You'll notice that coarsening alone is **faster than the naive kernel** (~1.4×) but **slower than tiling**. This is expected — the two techniques attack *different* bottlenecks:

| Technique | What it reduces | Helps with |
|---|---|---|
| Coarsening (this version) | Redundant loads of **A** | A is loaded once, reused across COARSEN columns |
| Tiling (parallel opt 2) | Redundant loads of **both A and B** | Entire tile loaded once into fast SMEM, reused by all threads |

Tiling addresses the bigger bottleneck (both matrices, not just A), so it wins on its own. Coarsening's value isn't to *replace* tiling — it's to **stack on top of it**. The combination (tiling + coarsening) is how high-performance matmul kernels approach cuBLAS-level throughput: the tile handles shared reuse across threads, and coarsening handles register-level reuse within each thread.

> **Exercise:** combine the tiled kernel from `parallel_optimized2.py` with the coarsening pattern above — each thread processes `COARSEN` output columns from the same shared-memory tiles. That's the version worth benchmarking against cuBLAS. Tune `COARSEN` carefully: too large spills registers into slow local memory, hurting occupancy and throughput.

### E. CUDA streams & pinned memory  —  verdict: limited for one matmul

**Pinned (page-locked) host memory** makes host↔device copies faster, because the driver can DMA it directly. It's an easy, low-risk transfer win whenever you copy large arrays:

```python
A_pinned = cuda.pinned_array(A.shape, dtype=A.dtype)
A_pinned[:] = A
d_A = cuda.to_device(A_pinned)   # faster transfer than from a normal (pageable) array
```

**Streams** let independent operations overlap — e.g. copying the next chunk of data while the current chunk is being computed, or running several kernels at once:

```python
stream = cuda.stream()
d_A = cuda.to_device(A, stream=stream)
kernel[grid, block, stream](d_A, ...)   # queued on the same stream
```

For a **single dense matmul**, streams give little: the kernel needs the *entire* `A` and `B` before it can start, so there is nothing to overlap the transfer with. Streams pay off when the work splits into independent chunks (a pipeline) or when you run many kernels — e.g. tile `C`'s rows into chunks and overlap copying chunk `i+1`'s inputs with computing chunk `i`. That's a meaningful project extension, but more complex than it is worth for this single-call example.


---

# Final comparison

All versions in one table, sorted from slowest to fastest:

In [ ]:
all_results.sort_values('time (s)', ascending=False)

In [ ]:
import matplotlib.pyplot as plt

# Split into CPU and GPU versions for readable scales
cpu_keywords = ['numpy', 'njit (CPU)', 'parallel=True']
is_cpu = all_results.index.map(lambda name: any(k in name for k in cpu_keywords))
df_cpu = all_results[is_cpu].sort_values('time (s)')
df_gpu = all_results[~is_cpu].sort_values('time (s)')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 8))

# Top chart: CPU versions (seconds scale)
df_cpu.plot.barh(ax=ax1, legend=False, color='steelblue')
for i, (val) in enumerate(df_cpu['time (s)']):
    ax1.text(val + 0.3, i, f'{val:.2f} s', va='center', fontsize=9)
ax1.set_xlabel('time (s)')
ax1.set_title('CPU versions  (seconds scale)')

# Bottom chart: GPU versions (milliseconds scale — much smaller numbers)
gpu_ms = df_gpu['time (s)'] * 1000
gpu_ms.plot.barh(ax=ax2, legend=False, color='darkorange')
for i, val in enumerate(gpu_ms):
    ax2.text(val + 1, i, f'{val:.1f} ms', va='center', fontsize=9)
ax2.set_xlabel('time (ms)')
ax2.set_title('GPU versions  (milliseconds scale)')

plt.tight_layout()
plt.show()

## Takeaways

- **Workflow:** analyze (find the costly step) → design → implement → evaluate *both* runtime and correctness. Record every version in one table so comparisons are honest.
- **Benchmark fairly:** warm up to exclude compilation; use wall-clock time consistently; for the GPU, time phases with CUDA events (not the deprecated `nvprof`).
- **GPU optimizations that helped here:** coalesced access (thread→data mapping), explicit memory management (transfer once, reuse), shared-memory tiling (cut slow GMEM reads), and 32-bit types.
- **Techniques that didn't help this problem:** grid-stride loops and streams — useful tools, wrong fit for a single dense matmul. Knowing *when not* to apply an optimization is as important as knowing the optimization.
- **Reality check:** these hand-written kernels are for learning. In production, `A @ B` (BLAS) on CPU or CuPy/cuBLAS on GPU will usually beat them. Numba's value is custom kernels that *aren't* a standard library call.

## References
- Numba CUDA — https://numba.readthedocs.io/en/stable/cuda/index.html
- Memory management (device arrays, pinned, streams) — https://numba.readthedocs.io/en/stable/cuda/memory.html
- Shared memory & examples (tiled matmul) — https://numba.readthedocs.io/en/stable/cuda/examples.html
- CUDA events / timing — https://numba.readthedocs.io/en/stable/cuda-reference/host.html